try fitting alpha vs pitch acc to watanabe et al model

In [ ]:
from betata.resonator_studies.resonator import load_resonators
resonators = load_resonators()

In [ ]:
selected_resonators = []
for resonator in resonators:
    l_kin, n_sq, n_sq_err = resonator.l_kin, resonator.N_sq, resonator.N_sq_err
    if resonator.type == "CPW" and None not in [l_kin, n_sq, n_sq_err]:
        selected_resonators.append(resonator)

In [ ]:
from betata import plt
plt.figure(figsize=(10, 10))
plt.scatter([res.pitch for res in selected_resonators], [res.alpha_bare for res in selected_resonators])

In [ ]:
max([res.alpha_bare for res in resonators])

In [ ]:
from dataclasses import dataclass
from scipy.constants import mu_0, epsilon_0
from scipy.special import ellipk
import numpy as np

@dataclass
class CPWModel:
    """ inductance and capacitance are calculated assuming zero thickness """

    length: float = None
    pitch: float = None
    width: float = None
    thickness: float = None
    lambda_eff: float = 1.8e-6
    epsilon_eff: float = (1 + 10.4) / 2 # sapphire, approximately accounting for anisotropy, 10.4 is geometric mean of 9.4 and 11.5

    @property
    def b(self):
        return self.width + 2 * self.pitch

    @property
    def k(self):
        return self.width / (self.b)

    @property
    def kprime(self):
        return np.sqrt(1 - self.k ** 2)

    @property
    def g(self):
        k, kprime = self.k, self.kprime
        t, w, p = self.thickness, self.width, self.pitch
        prefactor = (1 / (2 * kprime ** 2 * ellipk(k) ** 2))
        term1 = -np.log(t / (4 * w))
        term2 = -k * np.log(t / (4 * self.b))
        term3 = (1 + k) * np.log(p / (w + p))
        return prefactor * (term1 + term2 + term3)

    @property
    def Llm(self):
        """ magnetic inductance per unit length """
        return (mu_0 / 4) * (ellipk(self.kprime) / ellipk(self.k))

    @property
    def Llk(self):
        """ kinetic inductance per unit length """
        if self.thickness == 0:
            return 0
        return (mu_0 * self.lambda_eff ** 2 * self.g) / (self.thickness * self.width)

    @property
    def l_kin(self):
        """ total kinetic inductance """
        # factor of 0.5 to convert distributed inductance to effective modal inductance
        return self.Llk * self.length * 0.25

    @property
    def l_geom(self):
        """ total geometric inductance """
        # factor of 0.5 to convert distributed inductance to effective modal inductance
        return self.Llm * self.length * 0.5

    @property
    def alpha(self):
        """ kinetic inductance fraction """
        ki = self.l_kin
        return ki / (self.l_geom + ki)

    @property
    def Cl(self):
        """ capacitance per unit length"""
        return 4 * self.epsilon_eff * epsilon_0 * (ellipk(self.k) / ellipk(self.kprime))

    @property
    def f_anal(self):
        """ analytical resonant frequency, geometric only """
        return 1 / (4 * self.length * np.sqrt(self.Llm * self.Cl))

    #@property
    #def N_sq(self):
    #    """ watanabe model for what we have been calling N_sq obtained from axiem """
    #    return (self.g / (self.width)) * self.length

    #@property
    #def N_sq(self):
    #    """ model acc to chat gpt for what we have been calling N_sq from axiem """
    #    prefactor = (np.pi / (4 * self.b)) * self.length
    #    return prefactor * (ellipk(self.kprime) / ellipk(self.k))

Theory curves for alpha vs pitch vs thickness

In [ ]:
def g_factor(w, s, t):
    """ w: width, s: pitch, t: thickness """
    b = w + 2 * s
    k = w / b

    prefactor = (1 / (2 * k ** 2 * ellipk(k) ** 2))
    term1 = -np.log(t / (4 * w))
    term2 = -k * np.log(t / (4 * b))
    term3 = ((2 * (w + s)) / b) * np.log(s / (w + s))

    return prefactor * (term1 + term2 + term3)

def lk_model(thickness, pen_depth, prefactor):
    """ """
    return (mu_0 * prefactor * pen_depth) / (np.tanh(thickness / pen_depth))
    #return (mu_0 * prefactor * pen_depth ** 2) / thickness

def alpha_model(w, s, t, pen_depth, t0_model=True):
    """ """
    k = w / (w + 2 * s)
    kprime = np.sqrt(1 - k ** 2)

    l_geom = (mu_0 / 4) * (ellipk(kprime) / ellipk(k))

    l_kin_prefactor = prefactor1(w, s) if t0_model else prefactor2(w, s, t)
    l_kin = lk_model(t, pen_depth, l_kin_prefactor)

    return l_kin / (l_kin + l_geom * 0.5)

def prefactor1(w, s):
    """ """
    b = w + 2 * s
    k = w / b
    kprime = np.sqrt(1 - k ** 2)

    prefactor = (np.pi / (4 * b))
    return prefactor * (ellipk(kprime) / ellipk(k))

def prefactor2(w, s, t):
    """ """
    return g_factor(w, s, t) / (w)

def error_fn(params, resonators, t0_model):
    """
    t0_model: use zero-thickness model for KI 
    """
    pen_depth = params["pen_depth"].value
    data = np.array([res.alpha_bare for res in resonators])
    model = []
    for resonator in resonators:
        w = resonator.width
        s = resonator.pitch
        t = resonator.film_thickness
        length = resonator.length
        alpha = alpha_model(w, s, t, pen_depth, t0_model=t0_model)
        model.append(alpha)
    model = np.array(model)
    return data - model

In [ ]:
from scipy.optimize import fsolve
import lmfit

use_zero_t_model = True
CHAR_IMP = 50
EPSILON_EFF = (1 + 10.4) / 2

init_params = lmfit.Parameters()
init_params.add("pen_depth", value=1.6e-6, min=0)

fit_result = lmfit.minimize(error_fn, init_params, args=(selected_resonators, use_zero_t_model))
print(lmfit.fit_report(fit_result))

plt.figure(figsize=(12, 12))

pen_depth_result = fit_result.params["pen_depth"].value

pitches_interp = np.linspace(2e-6, 16e-6, 101)

def imp(w, s, h=530e-6):
    """ assume char imp 50 ohm """
    b = w + 2 * s
    k = w / b
    kprime = np.sqrt(1 - k ** 2)
    k3 = np.tanh((np.pi * w) / (4 * h)) / np.tanh((np.pi * b) / (4 * h))
    k3prime = np.sqrt(1 - k3 ** 2)

    epsilon_eff = EPSILON_EFF
    prefactor = (60 * np.pi) / np.sqrt(epsilon_eff)

    return prefactor / ((ellipk(k) / ellipk(kprime)) + (ellipk(k3) / ellipk(k3prime)))


widths_interp = []
for pitch in pitches_interp:
    width_fn = lambda w : CHAR_IMP - imp(w, pitch)
    width = fsolve(width_fn, pitch * 2)[0]
    widths_interp.append(width)

#print(imp(4.76e-6, 2e-6))
#print(imp(33.46e-6, 14e-6))
#print(imp(14.28e-6, 6e-6))
#print(imp(23.9e-6, 10e-6))
#print(imp(19.12e-6, 8e-6))
#print(imp(28.68e-6, 12e-6))
#print(imp(9.52e-6, 4e-6))
#print(imp(38.24e-6, 16e-6))

thicknesses = {res.film_thickness for res in selected_resonators}
for thickness in thicknesses:
    res_thickness = [res for res in selected_resonators if res.film_thickness == thickness]
    res_thickness_pitches = [res.pitch for res in res_thickness]
    res_thickness_alphas = [res.alpha_bare for res in res_thickness]
    plt.scatter(res_thickness_pitches, res_thickness_alphas)
    alphas_model = []
    for i in range(len(pitches_interp)):
        alpha_sample = alpha_model(widths_interp[i], pitches_interp[i], thickness, pen_depth_result, t0_model=use_zero_t_model)
        alphas_model.append(alpha_sample)
    plt.plot(pitches_interp, alphas_model)

In [ ]:
res_to_plot = []
for res in selected_resonators:
    if -np.inf < res.film_thickness < np.inf:
        res_to_plot.append(res)

model_resonators = []
for res in res_to_plot:
    model_res = CPWModel(length=res.length, pitch=res.pitch, width=res.width, thickness=res.film_thickness)
    #model_res.lambda_eff = 0.89e-6
    model_resonators.append(model_res)

In [ ]:
from betata import plt

plt.figure(figsize=(10, 10))

plt.scatter([res.pitch for res in res_to_plot], [res.alpha_bare for res in res_to_plot])
#plt.scatter([res.pitch for res in model_resonators], [res.f_anal for res in model_resonators], c="r")

In [ ]:
plt.scatter([res.pitch for res in model_resonators], [res.Cl * res.length for res in model_resonators], c="r")